# Energy Fraud Detection — Colab Runner

**Before running**: set the runtime to **GPU** via `Runtime → Change runtime type → T4 GPU`.

## Step 1 — Install dependencies and restart runtime

Run this cell, wait for it to finish, then go to **`Runtime → Restart session`**.
After the restart, **skip this cell** and continue from Step 2.

In [ ]:
!pip install \
    'scikit-learn==1.4.2' \
    'imbalanced-learn==0.12.4' \
    'pytorch-lightning==1.9.5' \
    'torchmetrics==0.11.4' \
    polars pyarrow tqdm \
    xgboost shap

print('\nAll done — go to Runtime → Restart session, then continue from Step 2.')

## Step 2 — Clone the repo

In [ ]:
!git clone https://github.com/Vict5/smart-meter-fraud.git

import os, sys
REPO_PATH = '/content/smart-meter-fraud'
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

print('Working directory:', os.getcwd())

## (Optional) Refresh repo — re-clone to pick up latest fixes

Run this if you need to pull the latest code changes without starting a new Colab session.
Move back to `/content` first so Python isn't inside the directory being deleted.

In [ ]:
import os, sys, importlib

os.chdir('/content')
!rm -rf /content/smart-meter-fraud
!git clone https://github.com/Vict5/smart-meter-fraud.git

REPO_PATH = '/content/smart-meter-fraud'
os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# Reload modules so updated source code takes effect without a full runtime restart
import src.features.build_features as bf
importlib.reload(bf)

print('Repo refreshed. Re-run Step 3 (upload) and Step 4 (preprocess) next.')

## Step 3 — Upload raw data files

Upload all 6 raw CSV files: `CONSUMI.csv`, `ANAGRAFICA.csv`, `LAVORI.csv`, `INTERRUZIONI.csv`, `PAROLE_DI_STATO.csv`, `LABELS.csv`.

In [ ]:
import os
from google.colab import files

os.makedirs('data/raw', exist_ok=True)

print('Select all 6 raw CSV files in the upload dialog...')
uploaded = files.upload()

for filename, data in uploaded.items():
    dest = os.path.join('data/raw', filename)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  saved -> {dest}')

print('\nFiles in data/raw/:', os.listdir('data/raw'))

## Step 4 — Preprocess raw data

Reads from `data/raw/` and writes cleaned CSV files to `data/processed/`.

In [ ]:
import os, pathlib

os.makedirs('data/processed', exist_ok=True)
os.makedirs('src/models/scalers', exist_ok=True)

from src.features.build_features import PreprocessDataset

preprocessor = PreprocessDataset(pathlib.Path('data/raw'))
preprocessor.preprocess_all()

print('Preprocessing done. Files in data/processed/:', os.listdir('data/processed'))

## Step 5 — (Optional) Restore cached models from Drive

Skip on the first run. On later sessions, uncomment and run to restore
previously trained models and skip ~1 hour of autoencoder training.

In [ ]:
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# CACHE = '/content/drive/MyDrive/smart-meter-fraud-cache'
# shutil.copytree(f'{CACHE}/models',              'models',                   dirs_exist_ok=True)
# shutil.copytree(f'{CACHE}/embeddings',          'data/embeddings',          dirs_exist_ok=True)
# shutil.copytree(f'{CACHE}/combined_embeddings', 'data/combined_embeddings', dirs_exist_ok=True)
# print('Models restored from Drive.')

## Step 6 — Enable inline plots

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

## (Optional) Override config hyperparameters

Run this **before Step 7** if you want a quick smoke-test with fewer epochs.

In [ ]:
# from src.models.config import Config
# Config.N_EPOCHS_AE = 50
# Config.N_EPOCHS_DNN = 100

## Step 7 — Run the pipeline

First run trains all 5 autoencoders (~1 hr on GPU). Later runs detect saved files and skip straight to classification.

| Mode | Description |
|------|-------------|
| `XGBoost` | 4-fold CV XGBoost ensemble (recommended) |
| `RandomForest` | 4-fold CV Random Forest ensemble |
| `DNN_Classifier` | 4-fold CV MLP ensemble |
| `Binary_Classifier` | Threshold anomaly detection (requires `ONLY_REGULAR=True` in config) |

In [ ]:
from src.pipeline import main
main(mode='XGBoost')

In [ ]:
# Other modes:
# main(mode='RandomForest')
# main(mode='DNN_Classifier')
# main(mode='Binary_Classifier')  # only when ONLY_REGULAR=True in config

## Step 8 — View predictions per customer

Loads saved embeddings and XGBoost models, prints each customer's Supply_ID,
predicted class, and class probabilities.

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import os

# Load combined embeddings from disk
combined = np.load('data/combined_embeddings/only_regular/combined_embeddings.npy')

# Load Supply_IDs in the same sorted order as combined_embeddings
labels_df = pd.read_csv('data/processed/LABELS.csv', sep='\t', encoding='utf-16')
labels_df = labels_df.sort_values('Supply_ID').reset_index(drop=True)
supply_ids = labels_df['Supply_ID'].values

# Load saved XGBoost folds and soft-vote
models_dir = 'models/XGBoost/only_regular'
xgb_models = []
for fold in range(4):
    m = xgb.XGBClassifier()
    m.load_model(os.path.join(models_dir, f'fold_{fold}.json'))
    xgb_models.append(m)

probas = np.mean([m.predict_proba(combined) for m in xgb_models], axis=0)
predictions = np.argmax(probas, axis=1)

# Build results table
label_map = {0: 'Anomalia', 1: 'Frode', 2: 'Regolare'}
results = pd.DataFrame({
    'Supply_ID':     supply_ids,
    'Prediction':    [label_map[p] for p in predictions],
    'Prob_Anomalia': np.round(probas[:, 0], 4),
    'Prob_Frode':    np.round(probas[:, 1], 4),
    'Prob_Regolare': np.round(probas[:, 2], 4),
})

print(results['Prediction'].value_counts().to_string())
print()
print(results.to_string(index=False))

In [ ]:
# Compare against true labels (if available)
true_label_map = {0: 'Anomalia', 1: 'Frode', 2: 'Regolare'}
results['True_Label'] = labels_df['CLUSTER'].map(true_label_map).values
results['Correct'] = results['Prediction'] == results['True_Label']

wrong = results[~results['Correct']]
print(f'Misclassified: {len(wrong)} / {len(results)}')
print(wrong[['Supply_ID', 'True_Label', 'Prediction', 'Prob_Frode']].to_string(index=False))

## Step 9 — (Optional) Save trained models to Drive

Run after the first successful pipeline run to cache models for future sessions.

In [ ]:
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# CACHE = '/content/drive/MyDrive/smart-meter-fraud-cache'
# shutil.copytree('models',                   f'{CACHE}/models',              dirs_exist_ok=True)
# shutil.copytree('data/embeddings',          f'{CACHE}/embeddings',          dirs_exist_ok=True)
# shutil.copytree('data/combined_embeddings', f'{CACHE}/combined_embeddings',  dirs_exist_ok=True)
# print('Models saved to Drive.')